In [1]:
import model_utils_shared  
from rasoul_seed import run_seed_clustering_pipeline
import numpy as np
import pandas as pd
import config 

/home/rass/Desktop/SocialScience-ConceptIntegration/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
neg_path = "datasets/processed_datasets/train_negative_pairs.csv"
pos_path = "datasets/processed_datasets/train_positive_pairs.csv"

# Load dataframe
df = model_utils_shared.load_and_prepare_data(pos_path, neg_path, balance=False)
print(f"Loaded dataframe with {len(df)} pairs.")

Loading datasets...
Positive samples: 3587
Negative samples (total): 2884723
Negative samples (used as is): 2884723
Loaded dataframe with 2888310 pairs.


In [3]:
models_to_test = config.MODELS
all_seed_clustering_results = []
random_seed = 42
n_initial_seeds = [10, 25, 50, 100, 250, 500]
n_random_trials = 5
similarity_thresholds = np.arange(0.1, 0.9, 0.05)

In [ ]:
for model_name, model_info in models_to_test.items():
    print(f"\nLoading model: {model_name} | model type: {model_info['type']}")
    model = model_utils_shared.load_model(model_name=model_name, model_type=model_info['type'])
    
    df_results, grouped, best_n_initial, best_threshold, best_f1_mean = run_seed_clustering_pipeline(
        df, 
        model,
        n_initial_seeds=n_initial_seeds,
        similarity_threshold=similarity_thresholds,
        random_state=random_seed,
        num_random_trials=n_random_trials
    )
    
    # Store results
    all_seed_clustering_results.append({
        'model_name': model_name,
        'df_results': df_results,
        'grouped': grouped,
        'best_n_initial': best_n_initial,
        'best_threshold': best_threshold,
        'best_f1_mean': best_f1_mean
    })
    
    # Display results for this model
    print("\n" + "="*80)
    print(f"MODEL: {model_name}")
    print("="*80)
    print("\nINDIVIDUAL TRIAL RESULTS:")
    print("-"*80)
    print(f"BEST CONFIGURATION:")
    print(f"  Seeds: {int(best_n_initial)} | Threshold: {best_threshold:.2f} | Mean F1: {best_f1_mean:.4f}")
    print("="*80)


Loading model: all-mpnet-base-v2 | model type: sentence_transformer
Loading Model (all-mpnet-base-v2)...
Total unique terms: 4270


Batches: 100%|██████████| 134/134 [00:14<00:00,  9.44it/s]


  Unclustered (new seeds created): 0
--------------------------------------------------
Seeds:   10|Threshold:   0.10|Trial:  1|F1:  0.0138|Precision:  0.0069|Recall:  0.7326|Pos Acc:  0.7326|Neg Acc:  0.8697|Clusters:   10
  Unclustered (new seeds created): 0
--------------------------------------------------
Seeds:   10|Threshold:   0.10|Trial:  2|F1:  0.0100|Precision:  0.0050|Recall:  0.7037|Pos Acc:  0.7037|Neg Acc:  0.8271|Clusters:   10
  Unclustered (new seeds created): 0
--------------------------------------------------
Seeds:   10|Threshold:   0.10|Trial:  3|F1:  0.0138|Precision:  0.0070|Recall:  0.7268|Pos Acc:  0.7268|Neg Acc:  0.8712|Clusters:   10
  Unclustered (new seeds created): 0
--------------------------------------------------
Seeds:   10|Threshold:   0.10|Trial:  4|F1:  0.0131|Precision:  0.0066|Recall:  0.7154|Pos Acc:  0.7154|Neg Acc:  0.8663|Clusters:   10
  Unclustered (new seeds created): 0
--------------------------------------------------
Seeds:   10|Thre

In [ ]:
# Summary: Best configurations for all models
print("\n" + "="*100)
print("SUMMARY: BEST CONFIGURATIONS FOR ALL MODELS")
print("="*100)

summary_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    best_n_initial = result['best_n_initial']
    best_threshold = result['best_threshold']
    best_f1_mean = result['best_f1_mean']
    
    summary_list.append({
        'Model': model_name,
        'Best Seeds': int(best_n_initial),
        'Best Threshold': f"{best_threshold:.2f}",
        'Mean F1': f"{best_f1_mean:.4f}"
    })

summary_df = pd.DataFrame(summary_list)
print(summary_df.to_string(index=False))
print("="*100)


SUMMARY: BEST CONFIGURATIONS FOR ALL MODELS
                           Model  Best Seeds Best Threshold Mean F1
               all-mpnet-base-v2          10           0.50  0.1515
        dwulff/mpnet-personality          10           0.50  0.2599
allenai/scibert_scivocab_uncased          25           0.10  0.0075
               bert-base-uncased          25           0.10  0.0060
               all-mpnet-base-v2          10           0.50  0.1515
        dwulff/mpnet-personality          10           0.50  0.2599
allenai/scibert_scivocab_uncased          25           0.10  0.0075
               bert-base-uncased          25           0.10  0.0060


In [ ]:
# Combine individual results from all models and save to CSV
individual_results_list = []
for result in all_seed_clustering_results:
    model_name = result['model_name']
    df_results = result['df_results'].copy()
    df_results['model_name'] = model_name
    individual_results_list.append(df_results)

all_individual_results_df = pd.concat(individual_results_list, ignore_index=True)

# Save to CSV
output_path = "results/seed_clustering_individual_results.csv"
all_individual_results_df.to_csv(output_path, index=False)
print(f"\nSaved individual results to: {output_path}")
print(f"Total rows: {len(all_individual_results_df)}")
print(all_individual_results_df.head())


Saved individual results to: results/seed_clustering_individual_results.csv
Total rows: 64
                 model  threshold  trial  precision    recall        f1  \
0  SentenceTransformer        0.1      1   0.006945  0.732646  0.013759   
1  SentenceTransformer        0.1      2   0.005036  0.703652  0.010000   
2  SentenceTransformer        0.5      1   0.087654  0.580708  0.152316   
3  SentenceTransformer        0.5      2   0.086687  0.577641  0.150751   
4  SentenceTransformer        0.1      1   0.014422  0.656816  0.028224   

    pos_acc   neg_acc  num_initial_seeds         model_name  
0  0.732646  0.869729                 10  all-mpnet-base-v2  
1  0.703652  0.827119                 10  all-mpnet-base-v2  
2  0.580708  0.992484                 10  all-mpnet-base-v2  
3  0.577641  0.992433                 10  all-mpnet-base-v2  
4  0.656816  0.944186                 25  all-mpnet-base-v2  
